# Milestone 2 — LaTeX Processing

## Objectives

This milestone focuses on parsing LaTeX source files,
identifying document structure, and extracting relevant elements.

In [53]:
# Setup: locate repo root so imports and paths work from any working directory
import re
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
sys.path.insert(0, str(ROOT))

from src.utils.table_io import load_table, PROCESSED_DIR  # fallback loader

# Part II — Milestone 2: Hierarchical Parsing and Standardization

## 6. Parsing methodology

### 6.1 Main compilation path

The publication root is the `.tex` file containing `\\documentclass`. Follow `\\input` and `\\include` recursively, ignoring commands commented out in LaTeX comments. Parse only files reachable from the root; ignore redundant auxiliary `.tex` files.

Explain how the implementation handles:
- Root detection and multiple possible roots
- Relative paths and omitted `.tex` extensions
- Recursive includes and include cycles
- Comments and missing included files
- Versions without TeX sources (record as empty, do not silently skip)

**Implementation:** TODO

In [54]:
# TODO: Link or import the actual parser implementation from src/latex_parser/.
# Example:
# from src.latex_parser import find_root_tex, collect_compilation_files, parse_publication

### 6.2 Hierarchy and element extraction

Describe how the parser represents a document as a tree and assigns parent-child relationships. Required leaf granularity includes:
- Sentences from prose, without splitting at non-sentence periods in math, commands, abbreviations (e.g. `i.e.`, `et al.`, `Fig. 1`), URLs, or decimal numbers
- Block formulas (e.g. `equation` environments or `$$...$$`)
- Figures; tables are treated as a type of figure
- Itemized/enumerated lists as branching components with each item at the next level

Include acknowledgements and appendices, including unnumbered `\\section*` sections. Do not parse the references section into the hierarchy.

**Element kinds and hierarchy-level rules:** TODO

**Sentence boundary handling:** TODO

**Formula / figure / table handling:** TODO

### 6.3 Cleanup and math normalization

Explain how non-semantic LaTeX formatting is removed (examples in the specification include `\\centering`, placement options such as `[htpb]`, and `\\midrule`) while preserving semantic content. State the chosen unified representation for inline and block math.

**Cleanup rules:** TODO

**Inline math format:** TODO

**Block math format:** TODO

**Validation checks:** TODO

### 6.4 `\\bibitem` conversion and reference deduplication

Describe the programmatic conversion from `\\bibitem` definitions to standard BibTeX entries. Then explain cross-version deduplication:
- Identify entries referring to the same underlying reference, even if citation keys differ.
- Select one canonical citation key.
- Merge fields from duplicate entries instead of discarding information.
- Rewrite `\\cite{...}` commands to use the canonical key.

**Conversion method:** TODO

**Matching/canonical-key policy:** TODO

**Field union policy:** TODO

**Citation-key rewriting and tests:** TODO

### 6.5 Full-text content deduplication across versions

After cleanup, elements whose text content matches exactly across versions should share one `element_id`. The `hierarchy` table still records the parent-child edges separately for each version.

Explain:
- How content is normalized before comparison
- How IDs are generated and kept unique within a publication
- How identical content under different parents/versions is represented
- How root/document elements are handled

**Implementation:** TODO

## 7. Milestone 2 — output tables and validation

### 7.1 Expected tables

**`elements`** — one row per distinct cleaned element across all versions:
`element_id`, `arxiv_id`, `kind`, `content`

**`hierarchy`** — one row per node-parent edge for each version:
`arxiv_id`, `version`, `element_id`, `parent_id`

**`bibtex`** — unified BibTeX entries after deduplication and field merging:
`arxiv_id`, `citation_key`, `entry_type`, `title`, `authors`, `year`, `doi`, `journal`, `url`

### 7.2 Coverage and correctness checks

Before analysing the processed data, we verify that the output tables are complete and internally consistent. We run six checks:

- **Parsing success rate**: the share of paper versions that were parsed successfully, out of all versions (including empty and failed ones).
- **Version coverage**: how many versions each paper has, and how many of them have parsed content or are recorded as empty (no TeX source).
- **Deduplication effect**: the number of elements and BibTeX entries before and after deduplication.
- **Hierarchy integrity**: every `parent_id` in `hierarchy` exists in `elements`, and no version's tree contains a cycle.
- **Citation integrity**: each `citation_key` appears only once per paper in `bibtex`, and every `\cite` key in the text exists in `bibtex`.
- **Cross-version element reuse**: how many elements are shared by more than one version.

**Validation strategy and test cases:** All checks below are computed directly from the output tables, so they run again every time the book is built. The integrity checks end with an `assert`, so the notebook fails loudly if a rule is violated instead of silently reporting bad data.

In [55]:
ELEMENTS_PATH = PROCESSED_DIR / "elements.parquet"
HIERARCHY_PATH = PROCESSED_DIR / "hierarchy.parquet"
BIBTEX_PATH = PROCESSED_DIR / "bibtex.parquet"
MANIFEST_PATH = PROCESSED_DIR / "parse_manifest.csv"

elements = load_table(ELEMENTS_PATH)
hierarchy = load_table(HIERARCHY_PATH)
bibtex = load_table(BIBTEX_PATH)
parse_manifest = pd.read_csv(MANIFEST_PATH) if MANIFEST_PATH.exists() else pd.DataFrame()

for name, frame in [
    ("elements", elements),
    ("hierarchy", hierarchy),
    ("bibtex", bibtex),
    ("parse_manifest", parse_manifest)
]:
    print(f"{name}: {frame.shape}")
    if not frame.empty:
        display(frame.head())

Not found yet: D:\Project\Intro-ds\Intro-DS\data\processed\elements.parquet
Not found yet: D:\Project\Intro-ds\Intro-DS\data\processed\hierarchy.parquet
Not found yet: D:\Project\Intro-ds\Intro-DS\data\processed\bibtex.parquet
elements: (0, 0)
hierarchy: (0, 0)
bibtex: (0, 0)
parse_manifest: (0, 0)


In [56]:
results = {}

# Check A: every parent_id must exist as an element_id
if not hierarchy.empty and not elements.empty:
    known_ids = set(elements["element_id"])
    results["edges (total)"] = len(hierarchy)
    results["edges whose parent_id is missing"] = int((~hierarchy["parent_id"].isin(known_ids)).sum())

    # Check B: a version's tree must not contain a cycle
    cyclic = 0
    for (paper, version), group in hierarchy.groupby(["arxiv_id", "version"]):
        parent_of = dict(zip(group["element_id"], group["parent_id"]))
        for start in parent_of:
            seen, node = set(), start
            while node in parent_of:          # climb towards the root
                if node in seen:              # came back to a visited node -> cycle
                    cyclic += 1
                    break
                seen.add(node)
                node = parent_of[node]
            else:
                continue
            break                             # stop checking this version after one cycle
    results["versions containing a cycle"] = cyclic

# Check C: citation_key unique within each paper
if not bibtex.empty:
    dup_rows = bibtex.duplicated(subset=["arxiv_id", "citation_key"], keep=False).sum()
    results["bibtex rows (total)"] = len(bibtex)
    results["bibtex rows with duplicated key in same paper"] = int(dup_rows)

display(pd.Series(results, name="value").to_frame())

assert results.get("edges whose parent_id is missing", 0) == 0, "Some parent_id values do not exist"
assert results.get("versions containing a cycle", 0) == 0, "A hierarchy contains a cycle"
assert results.get("bibtex rows with duplicated key in same paper", 0) == 0, "Duplicate citation keys"

,value


In [57]:
if results:
    display(Markdown(
        "**Interpretation.** All three assertions passed, so every `parent_id` resolves to an element, "
        "no version tree has a cycle, and `citation_key` is unique per paper. Orphan parents or cycles would break "
        "tree reconstruction in Milestone 3, and duplicate keys would make `\\cite` resolution ambiguous."
    ))
else:
    print("No tables loaded: integrity checks did not run.")

No tables loaded: integrity checks did not run.


In [58]:
# Check D: every \cite key in elements must exist in bibtex for the same paper
cite_re = re.compile(r"\\cite[a-zA-Z]*\*?(?:\[[^\]]*\])*\{([^}]*)\}")

if not elements.empty and not bibtex.empty:
    valid_keys = bibtex.groupby("arxiv_id")["citation_key"].apply(set).to_dict()
    cited, unknown = 0, []
    for paper, text in zip(elements["arxiv_id"], elements["content"].fillna("")):
        for group in cite_re.findall(text):
            for key in (k.strip() for k in group.split(",")):
                cited += 1
                if key not in valid_keys.get(paper, set()):
                    unknown.append((paper, key))
    print(f"\\cite keys checked: {cited:,}; not found in bibtex: {len(unknown):,}")
    print(unknown[:10])
else:
    print("elements or bibtex is empty: \\cite check skipped.")

elements or bibtex is empty: \cite check skipped.


In [59]:
# Parsing success rate and coverage (needs the manifest)
if not parse_manifest.empty:
    success_rate = (parse_manifest["status"] == "parsed").mean()
    print(f"Parsing success rate: {success_rate:.1%} of {len(parse_manifest):,} versions")
    display(parse_manifest["status"].value_counts().to_frame("versions"))
    print("Versions per paper:")
    display(parse_manifest.groupby("arxiv_id")["version"].nunique().describe().to_frame("versions per paper"))
else:
    print("parse_manifest.csv not found: success rate and empty versions cannot be computed yet.")

# Cross-version element reuse (from hierarchy)
if not hierarchy.empty:
    versions_per_element = hierarchy.groupby(["arxiv_id", "element_id"])["version"].nunique()
    reused = (versions_per_element > 1).sum()
    print(f"Elements shared by 2+ versions: {reused:,} of {len(versions_per_element):,} "
        f"({reused / len(versions_per_element):.1%})")

parse_manifest.csv not found: success rate and empty versions cannot be computed yet.


In [60]:
# Deduplication: how many elements and bibtex entries were removed by the shared store
if not hierarchy.empty and not elements.empty:
    # "Before": every node occurrence in every version (one hierarchy row = one occurrence)
    elems_before = len(hierarchy)
    # "After": distinct elements in the shared store, excluding document roots
    elems_after = int((elements["kind"] != "document").sum())
    print(f"Elements: {elems_before:,} before -> {elems_after:,} after "
        f"({1 - elems_after / elems_before:.1%} reduction)")

if not bibtex.empty:
    bib_after = len(bibtex)
    if not parse_manifest.empty and "bibtex_raw_count" in parse_manifest.columns:
        # "Before": raw entries counted by the parser per version, summed
        bib_before = int(parse_manifest["bibtex_raw_count"].sum())
        print(f"BibTeX entries: {bib_before:,} before -> {bib_after:,} after "
            f"({1 - bib_after / bib_before:.1%} reduction)")
    else:
        print(f"BibTeX entries after dedup: {bib_after:,} "
            "(before-count unavailable: parse_manifest.csv has no bibtex_raw_count)")

### 7.3 Summary statistics and figures

This section summarizes the processed corpus. Every figure is generated from the output tables (`elements`, `hierarchy`, `bibtex`) and `parse_manifest.csv` each time the book is built, and each one is followed by a short interpretation.

1. **Elements by kind**: distinct elements in the shared store, split into `sentence`, `formula`, `figure`, `heading` and the other kinds.
2. **Deduplication effect**: element rows and BibTeX entries before and after deduplication.
3. **Version coverage and parse status**: versions per paper, and how many versions are parsed, empty (no TeX source) or failed.
4. **Hierarchy size and depth**: number of nodes per version and maximum tree depth per version.

**Caveats.**
- The parse success rate is counted per version, not per paper.
- Empty versions (no TeX source) are kept separate from parser failures, using distinct `status` values in `parse_manifest.csv`.
- The BibTeX "before" count needs the `bibtex_raw_count` column in the manifest.
- Papers with a single version cannot show cross-version deduplication, so the reduction depends on how many papers have several versions.
- Tree depth may be computed on a random sample of versions (cap of x,000, fixed seed); the interpretation under the depth figure states how many versions were used.

In [61]:
if not elements.empty and "kind" in elements.columns:
    kind_counts = elements["kind"].value_counts(dropna=False)
    ax = kind_counts.plot(kind="bar", figsize=(7, 4))
    ax.set_title("Distinct parsed elements by kind")
    ax.set_xlabel("Element kind")
    ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()

    top, share = kind_counts.index[0], kind_counts.iloc[0] / kind_counts.sum()
    display(Markdown(
        f"**Interpretation.** The store holds {kind_counts.sum():,} distinct elements across "
        f"{len(kind_counts)} kinds. The most frequent kind is `{top}` ({share:.1%}). "
        f"The rarest kind is `{kind_counts.index[-1]}` ({kind_counts.iloc[-1]:,} rows)."
    ))
else:
    print("elements table empty: skipped.")

elements table empty: skipped.


In [62]:
panels, notes = [], []

if not hierarchy.empty and not elements.empty:
    e_before = len(hierarchy)                              # every node occurrence in every version
    e_after = int((elements["kind"] != "document").sum())  # distinct elements in the shared store
    panels.append(("Elements", ["before", "after"], [e_before, e_after]))
    notes.append(f"Element rows drop from {e_before:,} to {e_after:,} "
                f"({1 - e_after / max(e_before, 1):.1%} reduction), so sentences unchanged across versions are stored once.")

if not bibtex.empty:
    b_after = len(bibtex)
    if not parse_manifest.empty and "bibtex_raw_count" in parse_manifest.columns:
        b_before = int(parse_manifest["bibtex_raw_count"].sum())
        panels.append(("BibTeX entries", ["before", "after"], [b_before, b_after]))
        notes.append(f"BibTeX entries drop from {b_before:,} to {b_after:,} "
                    f"({1 - b_after / max(b_before, 1):.1%} reduction) after merging duplicate keys.")
    else:
        panels.append(("BibTeX entries", ["after"], [b_after]))
        notes.append(f"BibTeX has {b_after:,} entries after dedup; the manifest has no `bibtex_raw_count`, so no before-count.")

if panels:
    fig, axes = plt.subplots(1, len(panels), figsize=(4.5 * len(panels), 4), squeeze=False)
    for ax, (title, labels, values) in zip(axes[0], panels):
        ax.bar(labels, values)
        ax.set_title(title)
        ax.set_ylabel("Count")
    plt.tight_layout()
    plt.show()
    display(Markdown("**Interpretation.** " + " ".join(notes)))
else:
    print("elements/hierarchy/bibtex empty: dedup figure skipped.")

elements/hierarchy/bibtex empty: dedup figure skipped.


In [63]:
if not parse_manifest.empty:
    per_paper = parse_manifest.groupby("arxiv_id")["version"].nunique()
    status = parse_manifest["status"].value_counts()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    per_paper.plot(kind="hist", bins=range(1, int(per_paper.max()) + 2), ax=axes[0], align="left")
    axes[0].set_title("Versions per paper")
    axes[0].set_xlabel("Number of versions")
    status.plot(kind="bar", ax=axes[1])
    axes[1].set_title("Version status (parsed / empty / failed)")
    axes[1].set_ylabel("Versions")
    plt.tight_layout()
    plt.show()

    papers_with_empty = parse_manifest.loc[parse_manifest["status"] != "parsed", "arxiv_id"].nunique()
    display(Markdown(
        f"**Interpretation.** {len(per_paper):,} papers have on average {per_paper.mean():.2f} versions "
        f"(max {per_paper.max()}). {status.get('parsed', 0) / len(parse_manifest):.1%} of versions parsed; "
        f"{papers_with_empty:,} papers contain at least one non-parsed (empty or failed) version."
    ))
else:
    print("parse_manifest.csv not found: coverage figure skipped.")

parse_manifest.csv not found: coverage figure skipped.


In [64]:
def max_depth(parent_of):
    """Longest root-to-node path in one version's tree (root = depth 0)."""
    cache = {}
    best = 0
    for start in parent_of:
        path, n = [], start
        while n in parent_of and n not in cache:
            path.append(n)
            n = parent_of[n]
        d = cache.get(n, 0)
        for p in reversed(path):
            d += 1
            cache[p] = d
        best = max(best, cache.get(start, 0))
    return best

if not hierarchy.empty:
    nodes = hierarchy.groupby(["arxiv_id", "version"]).size()

    SAMPLE_N = 2000  # cap on versions used for depth, to keep the book build fast
    keys = nodes.sample(min(SAMPLE_N, len(nodes)), random_state=42).index
    sample = hierarchy.merge(keys.to_frame(index=False), on=["arxiv_id", "version"])

    depths = [max_depth(dict(zip(g["element_id"], g["parent_id"])))
            for _, g in sample.groupby(["arxiv_id", "version"])]

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].hist(nodes, bins=40, log=True)
    axes[0].set_title("Nodes per version")
    axes[0].set_xlabel("Nodes")
    axes[1].hist(depths, bins=range(1, max(depths) + 2), align="left")
    axes[1].set_title(f"Max tree depth per version ({len(depths):,} of {len(nodes):,} versions)")
    axes[1].set_xlabel("Depth")
    plt.tight_layout()
    plt.show()

    display(Markdown(
        f"**Interpretation.** Nodes per version: median {int(nodes.median()):,}, "
        f"mean {nodes.mean():,.0f}, max {int(nodes.max()):,}. "
        f"Depth was computed on {len(depths):,} of {len(nodes):,} versions (fixed seed, reproducible): "
        f"median {int(np.median(depths))}, max {max(depths)}."
    ))
else:
    print("hierarchy empty: skipped.")

hierarchy empty: skipped.


## 8. Milestone 2 — performance benchmark

Measure on the CPU-only Google Colab testbed:
- End-to-end parser wall time
- Average processing time per publication
- Maximum RAM used
- Final output storage size

| Metric | Value | Measurement method / notes |
|---|---:|---|
| End-to-end parser wall time (s) | TODO | TODO |
| Average time per publication (s) | TODO | TODO |
| Maximum RAM (MiB/GiB) | TODO | TODO |
| Final output size (MiB/GiB) | TODO | TODO |

**Benchmark method:** TODO  
**Runtime configuration:** TODO


## 9. Milestone 2 — discussion and limitations

Discuss:
- Which LaTeX patterns were difficult to parse?
- How were nested sections, lists, formulas, figures, and tables handled?
- What were the main causes of parsing failures?
- Did cleanup improve cross-version deduplication?
- Were duplicate bibliography entries merged correctly and citation keys rewritten?
- What limitations remain for Milestone 3?

**Findings:** TODO

**Limitations and next steps:** TODO


# 10. Overall conclusion

Summarize what the group built in Milestones 1 and 2, the most important quality/performance findings, known limitations, and how the processed corpus will support Milestone 3.

**Conclusion:** TODO


## References

Add complete citations for documentation, libraries, APIs, and external material used in the implementation. At minimum, consult and cite the official project statements linked below.

1. Course Lab Book, *arXiv Project — Problem Statements*: https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/
2. Course Lab Book, *Project Milestone 1 — Data Crawling*: https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/
3. Course Lab Book, *Project Milestone 2 — Hierarchical Parsing and Standardization*: https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/
4. TODO — arXiv API / source access documentation used.
5. TODO — Semantic Scholar API documentation used.
6. TODO — libraries and other external sources used.